In [ ]:
!pip install ultralytics opencv-python

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np

In [ ]:
model = YOLO("yolov8n.pt")

In [ ]:
import cv2
cap = cv2.VideoCapture("car3.mp4")
ret, frame  = cap.read()
print(frame.shape)

In [ ]:
from google.colab.patches import cv2_imshow
cv2_imshow(frame)

In [ ]:
results = model(frame)
results[0].show()

In [ ]:
target_box = None
for box in results[0].boxes:
    cls = int(box.cls[0])
    if cls == 2:
        x1, y1, x2, y2 = box.xyxy[0]
        x1 = int(x1)
        y1 = int(y1)
        x2 = int(x2)
        y2 = int(y2)
        area = (x2 - x1) * (y2 - y1)
        if area > 2000 and area < 20000 and y1 < 1300:
            target_box = box
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0,255,0),
                3
            )
            break
cv2_imshow(frame)

In [ ]:
x1, y1, x2, y2 = target_box.xyxy[0]

prev_x = int(x1)
prev_y = int(y1)

tracker_confidence = float(target_box.conf[0])
motion_confidence = 0.75
trusted_source = "Tracker"
print("Bounding Box:")
print("x =", int(x1))
print("y =", int(y1))
print("width =", int(x2 - x1))
print("height =", int(y2 - y1))
print("Tracker Confidence =", tracker_confidence)
print("Motion Model Confidence =", motion_confidence)
print("Trusted Source =", trusted_source)

In [ ]:
current_x = int(x1)
current_y = int(y1)

vx = current_x - prev_x
vy = current_y - prev_y

predicted_x = current_x + vx
predicted_y = current_y + vy

print("Velocity X =", vx)
print("Velocity Y =", vy)

print("Predicted Next X =", predicted_x)
print("Predicted Next Y =", predicted_y)

In [ ]:
cap = cv2.VideoCapture("car3.mp4")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    "output.mp4",
    fourcc,
    fps,
    (width, height)
)

In [ ]:
import cv2
from ultralytics import YOLO
import math

model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture("car3.mp4")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    "output.mp4",
    fourcc,
    fps,
    (width, height)
)

ret, frame = cap.read()

if not ret:
    print("Video not found")
    exit()

results = model(frame)

target_box = None

for box in results[0].boxes:

    cls = int(box.cls[0])

    if cls == 2:

        x1, y1, x2, y2 = box.xyxy[0]

        width_box = x2 - x1
        height_box = y2 - y1

        area = width_box * height_box

        # select small distant car
        if 5000 < area < 30000:

            target_box = box
            break

x1, y1, x2, y2 = target_box.xyxy[0]

prev_x = int(x1)
prev_y = int(y1)

width_box = int(x2 - x1)
height_box = int(y2 - y1)

tracker_confidence = float(target_box.conf[0])

motion_confidence = 0.75

trusted_source = "Tracker"

print("Bounding Box:")

print("x =", int(x1))
print("y =", int(y1))

print("width =", width_box)
print("height =", height_box)

print("Tracker Confidence =", tracker_confidence)

print("Motion Model Confidence =", motion_confidence)

print("Trusted Source =", trusted_source)


while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame)

    best_box = None

    minimum_distance = 999999

    for box in results[0].boxes:

        cls = int(box.cls[0])

        # ONLY CAR
        if cls != 2:
            continue

        x1, y1, x2, y2 = box.xyxy[0]

        x1 = int(x1)
        y1 = int(y1)

        x2 = int(x2)
        y2 = int(y2)

        current_width = x2 - x1
        current_height = y2 - y1

        # STRICT SIZE MATCH
        width_difference = abs(current_width - width_box)
        height_difference = abs(current_height - height_box)

        # if size changes too much -> reject
        if width_difference > 35:
            continue

        if height_difference > 35:
            continue

        # DISTANCE FROM PREVIOUS POSITION
        distance = math.sqrt(
            (x1 - prev_x) ** 2 +
            (y1 - prev_y) ** 2
        )

        # STRICT DISTANCE LIMIT
        if distance > 120:
            continue

        # CHOOSE CLOSEST BOX ONLY
        if distance < minimum_distance:

            minimum_distance = distance
            best_box = box

    # TRACK ONLY IF SAME VEHICLE FOUND
    if best_box is not None:

        x1, y1, x2, y2 = best_box.xyxy[0]

        x1 = int(x1)
        y1 = int(y1)

        x2 = int(x2)
        y2 = int(y2)

        width_box = x2 - x1
        height_box = y2 - y1

        tracker_confidence = float(best_box.conf[0])

        current_x = x1
        current_y = y1

        vx = current_x - prev_x
        vy = current_y - prev_y

        predicted_x = current_x + vx
        predicted_y = current_y + vy

        if tracker_confidence > 0.5:
            trusted_source = "Tracker"
        else:
            trusted_source = "Motion Model"

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Tracker:{tracker_confidence:.2f}",
            (x1, y1 - 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Motion:{motion_confidence:.2f}",
            (x1, y1 - 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

        cv2.putText(
            frame,
            f"Source:{trusted_source}",
            (x1, y1 - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"PredX:{predicted_x} PredY:{predicted_y}",
            (x1, y2 + 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

        # UPDATE ONLY SAME CAR
        prev_x = x1
        prev_y = y1

    out.write(frame)

cap.release()
out.release()

print("Tracking Completed")